# This is a solution for assignment4

### 1-a-i

This happens when:

The query q is very similar in direction to key $k_j$ (high cosine similarity)
AND/OR key $k_j$ has much larger magnitude than other keys
AND/OR the query q has large magnitude, amplifying the differences in dot products

Essentially, $k_j$ must "match" the query q much better than all other keys do.

### 1-a-ii

Under these conditions, the output c ≈ $v_j$ (approximately equals the value vector $v_j$).

## 1-b 

The query vector should be:
$$q = M \cdot \frac{k_a + k_b}{\|k_a + k_b\|}$$

where M is a large positive scalar.
As M → ∞, the exponential terms dominate, so α_a → ½ and α_b → ½, while α_i → 0 for all other i. (note that $e^0$ = 1)

### 1-c-i

The query vector should be:
$$q = M \cdot \frac{\mu_a + \mu_b}{\|\mu_a + \mu_b\|}$$

where M is a large positive scalar.

Therefore: c ≈ ½v_a + ½v_b = ½(v_a + v_b).

This works because the low variance ensures the key vectors behave predictably like their orthogonal means, allowing us to use the same construction as in the previous part.

### 1-c-ii 

**Analysis of the Modified Covariance**

Let me first understand what's happening with the new covariance structure:

**For key vector k_a:**
- $Σ_a$ = αI + ½($μ_a$ $μ_a^T$)
- This adds extra variance in the direction of $μ_a$
- So $k_a$ ≈ $μ_a$ + noise, where the noise has large variance along $μ_a$ direction

**For other key vectors k_i (i ≠ a):**
- $Σ_i$ = αI (same as before)
- So $k_i$ ≈ $μ_i$ with small spherical noise

Expected Behavior of c Across Different Samples

Using the same query q from part (i):

**The problem:** $k_a$ now has highly variable magnitude while pointing roughly in the same direction as μ_a.

**Dot product analysis:**
- $k_a^T q$ ≈ $||k_a|| · μ_a^T q/||μ_a|| = ||k_a|| · M/||μ_a + μ_b||$
- $k_b^T q$ ≈ $||μ_b|| · M/||μ_a + μ_b|| = M/||μ_a + μ_b||$ (stable)
- Other $k_i^T q$ ≈ 0 (still negligible)

Qualitative Expectations

**High variability in attention weights:**
- When $||k_a||$ is large: $α_a >> α_b, so c ≈ v_a$ (mostly copying v_a)
- When $||k_a||$ is small: $α_a << α_b, so c ≈ v_b$ (mostly copying v_b)  
- When $||k_a||$ ≈ $||μ_a||$ = $1: α_a ≈ α_b ≈ ½, so c ≈ ½(v_a + v_b)$ (desired behavior)

**Increased variance in c:**
- Unlike part (i) where c was consistently ≈ $½(v_a + v_b)$
- Now c varies dramatically between samples, ranging from ≈ $v_a$ to ≈ $v_b$
- The output becomes unreliable and unpredictable

**Key insight:** This demonstrates a major drawback of single-headed attention - it's fragile to magnitude variations in key vectors, even when they point in the right direction. Small changes in key norms can completely change which values the attention focuses on, making the model's behavior unstable.

### 1-d-i

Design the two query vectors as:

$$q_1 = M \cdot \mu_a$$
$$q_2 = M \cdot \mu_b$$

where M is a large positive scalar.

*Brief Justification*

**For query q₁ = M · μₐ:**
- $k_a^T q₁ ≈ μₐ^T(M · μₐ) = M (since ||μₐ|| = 1)$
- $k_i^T q₁ ≈ μᵢ^T(M · μₐ) = 0 for i ≠ a$ (due to orthogonality)
- Therefore: c₁ ≈ vₐ (copies value vector vₐ)

**For query q₂ = M · μᵦ:**
- $k_b^T q₂ ≈ μᵦ^T(M · μᵦ) = M (since ||μᵦ|| = 1) $ 
- $k_i^T q₂ ≈ μᵢ^T(M · μᵦ) = 0 for i ≠ b $(due to orthogonality)
- Therefore: c₂ ≈ vᵦ (copies value vector vᵦ)

**Final multi-headed output:**
$$c = \frac{1}{2}(c_1 + c_2) ≈ \frac{1}{2}(v_a + v_b)$$

This approach is much more robust than the single-headed solution because each head specializes in attending to one specific value vector, and their combination achieves the desired averaging effect.

### 1-d-ii 

Using the same query vectors from part (i):
- q₁ = M · μₐ  
- q₂ = M · μᵦ

*Expected Behavior Across Different Samples*

**For c₁ (using q₁ = M · μₐ):**
- $k_a^T q₁ ≈ ||k_a|| · M$(since k_a points roughly in direction of μₐ)
- $k_i^T q₁ ≈ 0 for i ≠ a$ (due to orthogonality)
- Since ||k_a|| varies significantly due to the extra variance along μₐ direction:
  - **High variance in c₁**: Sometimes $c₁ ≈ v_a$ (when ||k_a|| is large), sometimes c₁ is more diffuse (when $||k_a|| $is small)

**For c₂ (using q₂ = M · μᵦ):**
- $k_b^T q₂ ≈ M $(stable, since $k_b ≈ μᵦ$ with small spherical noise)
- $k_i^T q₂ ≈ 0 for i ≠ b$
- **Low variance in c₂**: Consistently $c₂ ≈ v_b$ across samples

*Final Output c = ½(c₁ + c₂)*

**Qualitatively:** The output c will have **moderate variance** - much more stable than the single-headed case but not as stable as when all covariances were αI.

**Why this is better:** 
- c₂ acts as a "stabilizing anchor" that consistently contributes ½$v_b$
- Even when c₁ varies due to $k_a$'s unstable magnitude, the averaging with the stable c₂ prevents extreme outputs
- The variance in c is roughly half the variance that c₁ would have on its own

This demonstrates a key benefit of multi-headed attention: **robustness through diversification** - even if one head becomes unreliable due to key vector perturbations, other heads can compensate.

### 2-a-i


*Part (i): Proving $\mathbf{Z}_{\text{perm}} = \mathbf{P}\mathbf{Z}$*

Given:
- Input: $\mathbf{X}_{\text{perm}} = \mathbf{P}\mathbf{X}$ where $\mathbf{P}$ is a permutation matrix
- Properties: $\text{softmax}(\mathbf{P}\mathbf{A}\mathbf{P}^{\top}) = \mathbf{P}\ \text{softmax}(\mathbf{A})\ \mathbf{P}^{\top}$ and $\text{ReLU}(\mathbf{P}\mathbf{A}) = \mathbf{P}\ \text{ReLU}(\mathbf{A})$

**Step 1: Self-Attention Layer**

For the permuted input $\mathbf{X}_{\text{perm}} = \mathbf{P}\mathbf{X}$, the query, key, and value matrices become:

$$\mathbf{Q}_{\text{perm}} = \mathbf{X}_{\text{perm}}\mathbf{W}_Q = \mathbf{P}\mathbf{X}\mathbf{W}_Q = \mathbf{P}\mathbf{Q}$$

$$\mathbf{K}_{\text{perm}} = \mathbf{X}_{\text{perm}}\mathbf{W}_K = \mathbf{P}\mathbf{X}\mathbf{W}_K = \mathbf{P}\mathbf{K}$$

$$\mathbf{V}_{\text{perm}} = \mathbf{X}_{\text{perm}}\mathbf{W}_V = \mathbf{P}\mathbf{X}\mathbf{W}_V = \mathbf{P}\mathbf{V}$$

The attention scores for the permuted input are:
$$\mathbf{Q}_{\text{perm}}\mathbf{K}_{\text{perm}}^{\top} = (\mathbf{P}\mathbf{Q})(\mathbf{P}\mathbf{K})^{\top} = \mathbf{P}\mathbf{Q}\mathbf{K}^{\top}\mathbf{P}^{\top}$$

Since $\mathbf{P}$ is a permutation matrix, $\mathbf{P}^{\top} = \mathbf{P}^{-1}$.

Applying the softmax:
$$\text{softmax}\left(\frac{\mathbf{Q}_{\text{perm}}\mathbf{K}_{\text{perm}}^{\top}}{\sqrt{d}}\right) = \text{softmax}\left(\frac{\mathbf{P}\mathbf{Q}\mathbf{K}^{\top}\mathbf{P}^{\top}}{\sqrt{d}}\right)$$

Using the given property:
$$= \mathbf{P}\ \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d}}\right)\ \mathbf{P}^{\top}$$

Therefore, the self-attention output is:
$$\mathbf{H}_{\text{perm}} = \text{softmax}\left(\frac{\mathbf{Q}_{\text{perm}}\mathbf{K}_{\text{perm}}^{\top}}{\sqrt{d}}\right) \mathbf{V}_{\text{perm}}$$

$$= \mathbf{P}\ \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d}}\right)\ \mathbf{P}^{\top} \cdot \mathbf{P}\mathbf{V}$$

$$= \mathbf{P}\ \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^{\top}}{\sqrt{d}}\right)\ \mathbf{V} = \mathbf{P}\mathbf{H}$$

**Step 2: Feed-Forward Layer**

For the feed-forward layer with input $\mathbf{H}_{\text{perm}} = \mathbf{P}\mathbf{H}$:

$$\mathbf{Z}_{\text{perm}} = \text{ReLU}(\mathbf{H}_{\text{perm}}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2$$

$$= \text{ReLU}(\mathbf{P}\mathbf{H}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2$$

Note that $\mathbf{P}\mathbf{1} = \mathbf{1}$ (permuting a vector of ones gives a vector of ones), so:

$$= \text{ReLU}(\mathbf{P}\mathbf{H}\mathbf{W}_1 + \mathbf{P}\mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2$$

$$= \text{ReLU}(\mathbf{P}(\mathbf{H}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1))\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2$$

Using the given ReLU property:
$$= \mathbf{P}\ \text{ReLU}(\mathbf{H}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2$$

$$= \mathbf{P}\ \text{ReLU}(\mathbf{H}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{P}\mathbf{1}\cdot\mathbf{b}_2$$

$$= \mathbf{P}[\text{ReLU}(\mathbf{H}\mathbf{W}_1 + \mathbf{1}\cdot\mathbf{b}_1)\mathbf{W}_2 + \mathbf{1}\cdot\mathbf{b}_2]$$

$$= \mathbf{P}\mathbf{Z}$$

Therefore, $\mathbf{Z}_{\text{perm}} = \mathbf{P}\mathbf{Z}$.

### 2-a-ii

*Part (ii): Implications for Text Processing*

This result shows that **Transformers without positional embeddings are permutation-invariant** - they treat the input sequence as an unordered set rather than an ordered sequence.

**Why this is problematic for text processing:**

1. **Word order matters in language**: The meaning of a sentence fundamentally depends on the order of words. For example:
   - "The cat chased the dog" vs "The dog chased the cat"
   - "John gave Mary a book" vs "Mary gave John a book"

2. **Syntactic structure depends on position**: Grammar rules depend heavily on word order. Without positional information, the model cannot distinguish between different grammatical structures.

3. **Semantic relationships are positional**: Many semantic relationships (subject-verb-object, modifier-modified, etc.) are encoded through word position in most languages.

4. **Loss of sequential information**: Text is inherently sequential - removing position information eliminates crucial contextual cues that humans rely on for understanding.

**Conclusion**: This is exactly why position embeddings are essential in Transformer architectures - they break the permutation invariance and allow the model to process sequences as ordered data rather than unordered sets.

### 2-b-i

*Part (i): Do position embeddings solve the permutation invariance problem?*

**Yes, position embeddings solve the permutation invariance problem.**

**Explanation:**

1. **Breaking symmetry**: When we add position embeddings to word embeddings ($\mathbf{X}_{\text{pos}} = \mathbf{X} + \Phi$), tokens at different positions now have different representations even if they have identical word embeddings.

2. **Position-dependent representations**: Each token's final representation becomes a function of both its semantic content (word embedding) and its position in the sequence. This means identical words at different positions will have different vector representations.

3. **Non-commutative operations**: Once position information is encoded in the input representations, the self-attention mechanism will produce different attention patterns for different orderings of the same words, because the query, key, and value matrices will be different.

4. **Permutation sensitivity**: If we permute the input sequence after adding position embeddings, the resulting representations will be fundamentally different from simply permuting the output, because each token now carries position-specific information.

### 2-b-ii

**No, the sinusoidal position embeddings cannot be the same for two different positions.**

**Explanation:**

The position embeddings are **deterministic functions of position** $t$. For any two different positions $t_1 \neq t_2$, we have:

$\Phi_{(t_1, j)} \neq \Phi_{(t_2, j)} \text{ for at least some dimension } j$

**Why they must be different:**

1. **Different frequencies**: The sinusoidal functions use different frequencies $1/10000^{2i/d}$ for different dimensions $i$. 

2. **Unique encoding**: The combination of sine and cosine functions at different frequencies creates a unique "fingerprint" for each position $t$.

3. **Mathematical property**: For the sinusoidal encoding to be meaningful, it must provide a unique representation for each position. If two positions had identical embeddings, the model couldn't distinguish between them.

4. **Frequency analysis**: Since we use multiple frequency components (different values of $i$), even if some frequency components might coincidentally align for two different positions, the full vector across all dimensions will be different.

The sinusoidal design ensures that each position $t$ gets a unique embedding vector $\Phi_{(t, :)}$, which is essential for the model to maintain positional information throughout processing.

## Appendix

**What is a Covariance Matrix?**

A **covariance matrix** describes how much the components of a random vector vary together. For a d-dimensional random vector, it's a d×d matrix where:

- **Diagonal elements**: Variance of each component
- **Off-diagonal elements**: Covariance between different components

For a random vector **x** = [x₁, x₂, ..., xₐ]ᵀ, the covariance matrix Σ has entries:
- Σᵢⱼ = Cov(xᵢ, xⱼ) for i ≠ j (covariance between components)
- Σᵢᵢ = Var(xᵢ) (variance of component i)

**Understanding Σᵢ = αI**
The formula **Σᵢ = αI** means each covariance matrix is:

$$\Sigma_i = \alpha I = \alpha \begin{bmatrix} 1 & 0 & \cdots & 0 \\ 0 & 1 & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & 1 \end{bmatrix} = \begin{bmatrix} \alpha & 0 & \cdots & 0 \\ 0 & \alpha & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \alpha \end{bmatrix}$$

This means:
- **All diagonal elements = α**: Each component has variance α
- **All off-diagonal elements = 0**: Components are uncorrelated
- **Same for all i**: All key vectors have identical covariance structure

**Geometric Interpretation**
When α is "vanishingly small" (α ≈ 0):
- Each key vector kᵢ is distributed very tightly around its mean μᵢ
- The "cloud" of possible values forms a small sphere of radius √α around μᵢ
- Since α ≈ 0, kᵢ ≈ μᵢ with high probability

**Why This Matters**

This setup allows us to treat the random key vectors as if they were deterministic and equal to their means μᵢ, which makes the attention mechanism behave predictably - exactly like the orthogonal case from the previous problem.